# ORM Jacobian — CFD Dipole Component (ALBA-II, no feedback)

Validates `dRij_dbend_thick23_disp` (horizontal, with energy term) and the
dispersion Jacobian `dni_dqk_integral` when changing only the dipole component
of combined-function dipoles (CFDs) in ALBA-II.

Tensor convention: axis 0 = CFD, axis 1 = BPM, axis 2 = corrector.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("../scripts")))

from general_numerical import NumericalCalculation
from ACalORAT import AnaORM, plot_utils, math_utils, numerical

## Parameters

In [ ]:
calc = NumericalCalculation("ring_a2", {
    "direction" : "both",
    "elements"  : "dip",
    "linearize" : -1,
    "fringes"   : True,
    "dispersion": True,
    "step"      : 1e-5,
}, root=Path("../"))

calc.compute()

## Analytical — ORM Jacobian

In [ ]:
ring, ind = calc.ring, calc.ind

# Energy term: semi-numerical dORM/dEnergy (shape n_BPM x n_cor)
dRij_dE = numerical.quickdORMdEnergy(ring, ind)

energy = calc.results["energy"]  # shape (n_CFD,)

cORM_v = AnaORM.AnaORM(ring, "v", ind)
cORM_v.assign_optics()
ana_v = cORM_v.dRij_dk_energy_term(cORM_v.bpm, cORM_v.cor, cORM_v.CFD, dRij_dE["v"], energy)

cORM_h = AnaORM.AnaORM(ring, "h", ind)
cORM_h.assign_optics()
ana_h  = cORM_h.dRij_dbend_thick23_disp(cORM_h.bpm, cORM_h.cor, cORM_h.CFD)
ana_h += cORM_h.dRij_dk_energy_term(cORM_h.bpm, cORM_h.cor, cORM_h.CFD, dRij_dE["h"], energy)

## Results — ORM Jacobian

In [ ]:
num_v = calc.numerical("v")
num_h = calc.numerical("h")

err_v = math_utils.normalized_RMSE(num_v, ana_v, dims=(1, 2))
err_h = math_utils.normalized_RMSE(num_h, ana_h, dims=(1, 2))
print(f"Vertical   nRMSE per CFD (mitjana): {err_v.mean():.2f}%")
print(f"Horizontal nRMSE per CFD (mitjana): {err_h.mean():.2f}%")

plot_utils.plot_both_Zeus(num_v, num_h, ana_v, ana_h, xlabel="CFD")

## Analytical — Dispersion Jacobian

Change in horizontal dispersion at BPMs when varying the dipole component of each CFD.

`dni_dqk_integral(bpm, CFD)` returns shape `(n_BPM, n_CFD)` — transposed to `(n_CFD, n_BPM)` to match `di_dk`.

In [ ]:
# cORM_h already has optics; reuse it
# Output is (n_BPM, n_CFD) with default 2D broadcasting; transpose to (n_CFD, n_BPM)
ana_disp = cORM_h.dni_dqk_integral(cORM_h.bpm, cORM_h.CFD).T

## Results — Dispersion Jacobian

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

num_disp = calc.results["di_dk"]  # shape (n_CFD, n_BPM)

err_disp = math_utils.normalized_RMSE(num_disp, ana_disp, dims=(1,))
print(f"Dispersion nRMSE per CFD (mitjana): {err_disp.mean():.2f}%")

# Plot a few representative CFDs
n_show = min(4, num_disp.shape[0])
fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 3), sharey=True)
for ax, k in zip(axes, range(n_show)):
    ax.plot(num_disp[k], label="Numerical")
    ax.plot(ana_disp[k], '--', label="Analytical")
    ax.set_title(f"CFD {k}")
    ax.set_xlabel("BPM")
axes[0].set_ylabel(r"$d\eta_i / d\theta_k$ [m/rad]")
axes[0].legend()
plt.tight_layout()
plt.show()